In [22]:
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
import pytz  # ensure this is at the top of your script

root_folder = '../../../../../'
planning_data = os.path.join(root_folder,'planning')
output_data = os.path.join(planning_data,'9_output_reports')

aware_folder = os.path.join(root_folder,'3_Analytics_Warehouse_Backend')

column_names_location = os.path.join(aware_folder,'1_inputs')#'../../../1_inputs'
parquet_intermediate_location = os.path.join(aware_folder,'2_planning_processing_files')#'../../../2_planning_processing_files'
database_location = os.path.join(aware_folder,'5_database')#'../../../5_database'


In [23]:
# Columns
cols = pd.read_csv(os.path.join(column_names_location,"columns.csv"))["Column Names"].tolist()
cols_p2 = pd.read_csv(os.path.join(column_names_location,"p2_columns.csv"))["Column Names"].tolist()
sensors = pd.read_csv(os.path.join(planning_data,'Sensor_Mapping.csv'))
sensors

,Sensor Number,Sensor Names,Weekly Avg Mapping,Interior Hub In Out,Monthly Peds Hub in Out,Hub In/Out,Exit Entry,Floor,Location - Monthly Ped Report,Location - Retail Report,Consolidated Location,Zone Description - Retail Report,Zone Description - Sensor Map,Zone Number,Sensor Type,Oculus Grouping
0,1,Z01_T4-ChurchSt_RevDoor_IN,NaN,IN,0,IN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
1,2,Z01_T4-ChurchSt_RevDoor_OUT,NaN,OUT,0,OUT,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
2,3,Z01_T4-ChurchSt_SwingDoor_IN,NaN,IN,0,IN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
3,4,Z01_T4-ChurchSt_SwingDoor_OUT,NaN,OUT,0,OUT,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
4,5,Z01_T4-ChurchSt_WestEsc_UP,NaN,NaN,0,NaN,NaN,Street Level,NaN,NaN,NaN,NaN,H&M T4 Entrance from Church St,1,Phase 1 Sensor,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,269,Z42_CenterPassage_Stair_OUT,NaN,OUT,NaN,OUT,NaN,C2 Main Level,NaN,Center West Passage (Level C2),Center West Passage (Level C2),Center-west Passage between Oculus and West Co...,Center-west Passage between Oculus and West Co...,42,Phase 2 Sensor,Oculus Floor
269,270,Z43_NorthWestPassage_IN,NaN,IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
270,271,Z43_NorthWestPassage_OUT,NaN,OUT,NaN,OUT,NaN,C2 Main Level,NaN,NaN,NaN,Northwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN
271,272,Z43_SouthWestPassage_IN,NaN,IN,NaN,IN,NaN,C2 Main Level,NaN,NaN,NaN,Southwest Passage between Oculus and West Conc...,Northwest Passage between Oculus and West Conc...,43,Phase 2 Sensor,NaN


In [28]:
df = pd.read_parquet(os.path.join(database_location,"daily.parquet"), engine="pyarrow")
# df2 = pd.read_parquet(os.path.join(database_location,"database_cleaned_p2.parquet"), engine="pyarrow")

print("df: ",len(df))
# print("df2: ",len(df2))
df["From"] = pd.to_datetime(df["From"], errors="coerce")
df["day"] = df["From"].dt.floor("D")
df[df["day"] == "2026-02-16"]


df:  218684


,From,Location,Count,day
218429,2026-02-16,Z01_T4-ChurchSt_RevDoor_IN,1602.0,2026-02-16
218430,2026-02-16,Z01_T4-ChurchSt_SwingDoor_IN,1108.0,2026-02-16
218431,2026-02-16,Z02_T4-LibertySt_Doors_IN,4046.0,2026-02-16
218432,2026-02-16,Z02_T4-LibertySt_EastEsc46_IN,4241.0,2026-02-16
218433,2026-02-16,Z02_T4-LibertySt_Stair_IN,636.0,2026-02-16
...,...,...,...,...
218509,2026-02-16,Z29_1Train-C2-SWOculus_AllDoors_IN,2370.0,2026-02-16
218510,2026-02-16,Z29_C2_SWOculus_HubElev_INTOELEV,320.0,2026-02-16
218511,2026-02-16,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,124.0,2026-02-16
218512,2026-02-16,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,2838.0,2026-02-16


In [26]:
df1 = pd.read_parquet(os.path.join(database_location,"database_cleaned_p1.parquet"), engine="pyarrow")
df1["day"] = df1["From"].dt.floor("D")
df1[df1["day"] == "2026-02-16"]


,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z27_T3TransitLobby_Stair50_UP_OUT,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV,day
628992,2026-02-16 00:00:00,2026-02-16 00:05:00,0.0,0.0,0.0,1.0,8.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.0,0.0,2026-02-16
628993,2026-02-16 00:05:00,2026-02-16 00:10:00,0.0,0.0,1.0,0.0,4.0,0.0,1.0,3.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2026-02-16
628994,2026-02-16 00:10:00,2026-02-16 00:15:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,3.0,3.0,2.0,5.0,2026-02-16
628995,2026-02-16 00:15:00,2026-02-16 00:20:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,2.0,0.0,2026-02-16
628996,2026-02-16 00:20:00,2026-02-16 00:25:00,0.0,0.0,0.0,0.0,3.0,0.0,1.0,1.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,2026-02-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629275,2026-02-16 23:35:00,2026-02-16 23:40:00,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,2026-02-16
629276,2026-02-16 23:40:00,2026-02-16 23:45:00,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0,0.0,2026-02-16
629277,2026-02-16 23:45:00,2026-02-16 23:50:00,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,5.0,0.0,0.0,3.0,0.0,2026-02-16
629278,2026-02-16 23:50:00,2026-02-16 23:55:00,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2026-02-16


In [13]:
df["From"] = pd.to_datetime(df["From"], errors="coerce")  # or errors="raise" to fail on bad values
df["day"] = df["From"].dt.floor("D")
df[df["day"] == "2026-02-16"]


,From,Location,Count,day


In [ ]:
df[df["From"] == "2019-01-01"]

,From,Location,Count


In [6]:
in_sensors = (
    sensors
    .loc[sensors["Monthly Peds Hub in Out"] == "IN", "Sensor Names"]
    .tolist()
)
in_sensors

['Z02_T4-LibertySt_EastEsc46_IN',
 'Z02_T4-LibertySt_WestEsc45_IN',
 'Z02_T4-LibertySt_Stair_IN',
 'Z04_T4-SoConc-Entry_AllDoors_IN',
 'Z06_WestProj-C1-StairEsc_NorthStair_IN',
 'Z06_WestProj-C1-StairEsc_NorthEsc36_IN',
 'Z06_WestProj-C1-StairEsc_Elev_OUTOFELEV',
 'Z06_WestProj-C1-StairEsc_SouthEsc35_IN',
 'Z06_WestProj-C1-StairEsc_SouthStair_IN',
 'Z08_EastProj-C1-StairEsc_Elev_OUTOFELEV',
 'Z08_EastProj-C1-StairEsc_NorthEsc39_IN',
 'Z08_EastProj-C1-StairEsc_NorthStair_IN',
 'Z08_EastProj-C1-StairEsc_SouthStair_IN',
 'Z08_EastProj-C1-StairEsc_SouthEsc37_IN',
 'Z09_DeyStConc-Entry_NorthDoors_IN',
 'Z09_DeyStConc-Entry_SouthDoors_IN',
 'Z10_ETrain-HistoricEntry_Stair_IN',
 'Z11_T2-VeseySt_StreetDoors_IN',
 'Z12_T2-FultonSt_AllDoors_IN',
 'Z13_T1-Office-Entry_Gen-EastMostRevDoor_IN',
 'Z13_T1-Office-Entry_Conde-WestRevDoor_IN',
 'Z13_T1-Office-Entry_Conde-CenterSwingDoors_IN',
 'Z13_T1-Office-Entry_Conde-EastRevDoor_IN',
 'Z13_T1-Office-Entry_Gen-2ndEastMostRevDoor_IN',
 'Z13_T1-Office-E

In [10]:
df1["IN_Sensor_Sum"] = df1[in_sensors].sum(axis=1)
df1["day"] = df1["From"].dt.floor("D")

In [11]:
df1

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV,IN_Sensor_Sum,day
0,2020-02-24 00:00:00,2020-02-24 00:05:00,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,36.0,2020-02-24
1,2020-02-24 00:05:00,2020-02-24 00:10:00,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,34.0,2020-02-24
2,2020-02-24 00:10:00,2020-02-24 00:15:00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,31.0,2020-02-24
3,2020-02-24 00:15:00,2020-02-24 00:20:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,22.0,2020-02-24
4,2020-02-24 00:20:00,2020-02-24 00:25:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,2020-02-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625531,2026-02-03 23:35:00,2026-02-03 23:40:00,1.0,0.0,1.0,2.0,5.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,137.0,2026-02-03
625532,2026-02-03 23:40:00,2026-02-03 23:45:00,0.0,0.0,0.0,0.0,5.0,0.0,0.0,2.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,156.0,2026-02-03
625533,2026-02-03 23:45:00,2026-02-03 23:50:00,0.0,0.0,0.0,1.0,5.0,0.0,0.0,4.0,...,0.0,0.0,0.0,4.0,1.0,0.0,3.0,0.0,125.0,2026-02-03
625534,2026-02-03 23:50:00,2026-02-03 23:55:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,56.0,2026-02-03


In [17]:
df1_daily = df1.groupby(["day"], as_index=False).sum(["IN_Sensor_Sum"]).sort_values("day")

df1_daily = df1_daily[
    ["day", "IN_Sensor_Sum"]
]

df1_daily

filtered_2023_2025 = df1_daily[
    (df1_daily["day"] >= "2023-01-01") &
    (df1_daily["day"] <= "2025-12-31")
]

filtered_2023_2025

,day,IN_Sensor_Sum
1042,2023-01-01,104792.0
1043,2023-01-02,97635.0
1044,2023-01-03,149738.0
1045,2023-01-04,152931.0
1046,2023-01-05,152919.0
...,...,...
2133,2025-12-27,121255.0
2134,2025-12-28,127483.0
2135,2025-12-29,174616.0
2136,2025-12-30,186618.0


In [22]:
filtered_2023_2025.to_csv(os.path.join(output_data,"2023_2025_xovis_daily.csv"),index=False)

# output_data = os.path.join(root_folder,'9_output_reports')


In [19]:
output_data

'../../../../../9_output_reports'

In [ ]:
df1_corrected = df1.groupby(["Location", "date", "time"])["Count"].mean().reset_index()
df2_corrected = df2.groupby(["Location", "date", "time"])["Count"].mean().reset_index()


In [18]:
print(len(df_corrected))
print(len(df))

21517248
21581053


In [20]:
df_april_may_check= df_corrected 

df_april_may_check["date_sub"] = pd.to_datetime(df_corrected["date"])

df_april_may_check = (
    df_corrected[
        (df_corrected["date_sub"].dt.year == 2025) &
        (df_corrected["date_sub"].dt.month.isin([4, 5]))
    ]
    .groupby("date_sub")["Count"]
    .count()
    .reset_index()
)
df_april_may_check

,date_sub,Count
0,2025-04-01,12480
1,2025-04-02,12480
2,2025-04-03,12480
3,2025-04-04,12480
4,2025-04-05,12480
...,...,...
56,2025-05-27,12480
57,2025-05-28,12480
58,2025-05-29,12480
59,2025-05-30,12480


In [21]:
df_corrected

,Location,date,time,Count,date_sub
0,Z01_T4-ChurchSt_EastEsc_DOWN,2020-02-24,00:00:00,0.0,2020-02-24
1,Z01_T4-ChurchSt_EastEsc_DOWN,2020-02-24,00:30:00,2.0,2020-02-24
2,Z01_T4-ChurchSt_EastEsc_DOWN,2020-02-24,01:00:00,9.0,2020-02-24
3,Z01_T4-ChurchSt_EastEsc_DOWN,2020-02-24,01:30:00,0.0,2020-02-24
4,Z01_T4-ChurchSt_EastEsc_DOWN,2020-02-24,02:00:00,6.0,2020-02-24
...,...,...,...,...,...
21517243,Z43_SouthWestPassage_OUT,2025-12-02,21:30:00,55.0,2025-12-02
21517244,Z43_SouthWestPassage_OUT,2025-12-02,22:00:00,38.0,2025-12-02
21517245,Z43_SouthWestPassage_OUT,2025-12-02,22:30:00,37.0,2025-12-02
21517246,Z43_SouthWestPassage_OUT,2025-12-02,23:00:00,25.0,2025-12-02


In [22]:
df_corrected_rearranged = df_corrected[["Location", "Count", "date", "time"]]
df_corrected_rearranged

,Location,Count,date,time
0,Z01_T4-ChurchSt_EastEsc_DOWN,0.0,2020-02-24,00:00:00
1,Z01_T4-ChurchSt_EastEsc_DOWN,2.0,2020-02-24,00:30:00
2,Z01_T4-ChurchSt_EastEsc_DOWN,9.0,2020-02-24,01:00:00
3,Z01_T4-ChurchSt_EastEsc_DOWN,0.0,2020-02-24,01:30:00
4,Z01_T4-ChurchSt_EastEsc_DOWN,6.0,2020-02-24,02:00:00
...,...,...,...,...
21517243,Z43_SouthWestPassage_OUT,55.0,2025-12-02,21:30:00
21517244,Z43_SouthWestPassage_OUT,38.0,2025-12-02,22:00:00
21517245,Z43_SouthWestPassage_OUT,37.0,2025-12-02,22:30:00
21517246,Z43_SouthWestPassage_OUT,25.0,2025-12-02,23:00:00


In [24]:
df_corrected_rearranged.to_parquet(os.path.join(database_location,"database_cleaned_long.parquet"), engine="pyarrow", index=False)